# In this file, Im just trying to implement TFHE from scratch, just to learn it.
#### Parameters used throughout the notebook:
- p (Plaintext modulus): 2^3 = 8; range of cleartext inputs: [-4, -3, -2, -1 ,0, 1, 2, 3]
- q (ciphertext modulus): 2^32*; range after encoding: [-2^31, 2^31)

> *This is why all dtypes are set to np.int32.

In [1]:
import numpy as np

# LWE

In [2]:
# creating the config class
class LWEConfig:
    '''
        This class defines the configurations for encryption
    '''
    def __init__(self, p, q, noise_level, dimension):
        '''
            p: plaintext modulus
            q: ciphertext modulus
            noise_level: The noise level (used as standard deviation for generating the noise)
            dimension: m (number of elements in the sk)
        '''
        self.p = p
        self.q = q
        self.sigma = noise_level
        self.n = dimension

In [3]:
# creating the class to generate plaintext
class LWEPlaintext:
    '''
        This class contains the plaintext to be encrypted.
    '''
    def __init__(self, m):
        '''
            This class (for now), just stores the message wrapping it in a class. It might contain code for encoding it too later but i dont know
        '''
        self.m = m
        
    def __str__(self):
        return str(self.m)
    
    def __repr__(self):
        return str(self.m)

In [4]:
# utility functions
# INT32_MIN = np.iinfo(np.int32).min
# INT32_MAX = np.iinfo(np.int32).max
def generate_uniform_sample(config: LWEConfig):
    '''
        Generate an array of {size} elements using a uniform random distribution in the required range
    '''
    # get the limits
    low = (config.q//2) * -1
    high = (config.q//2) # not subtracting -1 because high is exclusive in random.randint
    # get the random array
    a = np.random.randint(low=low, high=high, size=config.n, dtype=np.int32)
    return a

def generate_normal_sample(config:LWEConfig):
    '''
        This function generates the normal sample required for encryption within the given range. Since this is LWE, we only need 1 element here as m = 1.
    '''
    ei = np.int32(np.random.normal(loc=0.0, scale=config.sigma) * (config.q//2))
    return ei

def encode_plaintext(m, config):
    '''
        This function encodes the message before its enrypted.
    '''
    # get the delta
    delta = config.q//config.p
    encoded = np.int32(delta * m)
    return LWEPlaintext(encoded)

def decode_plaintext(m, config):
    '''
        This function decodes the decrypted plaintext to remove the noise and get the final output
    '''
    # get the delta
    delta = config.q/config.p
    # round to remove the noise
    decoded = int(np.rint(m/delta))
    # need to take a mod wrt p centered around 0 to get the final output
    final = ((decoded + (config.p/2)) % config.p) - (config.p/2)
    return final

In [5]:
from __future__ import annotations
class LWECiphertext:
    '''
        This class defines the ciphertext.
    '''
    def __init__(self, a, b):
        self.a = a
        self.b = b

    def add_ciphertext(self, other: LWECiphertext):
        '''
            This function adds a given ciphertext to the current ciphertext and returns a new ciphertext.
        '''
        # adding their as and bs
        added_a = np.add(self.a, other.a, dtype=np.int32)
        added_b = np.add(self.b, other.b, dtype=np.int32)

        # # taking mods just to see if this works, but we have to take mod wrt q
        # q = config.q
        # added_a = ((added_a + (q//2)) % q) - (q//2)
        # added_b = ((added_b + (q//2)) % q) - (q//2) # im getting overflow errors when doing this

        return LWECiphertext(added_a, added_b)
    
    def sub_ciphertext(self, other:LWECiphertext):
        '''
            This function subtracts the other ciphertext from the current one.
        '''
        sub_a = np.subtract(self.a, other.a, dtype=np.int32)
        sub_b = np.subtract(self.b, other.b, dtype=np.int32)
        return LWECiphertext(sub_a, sub_b)
    
    def mul_plaintext(self, other:np.int32):
        '''
            Multiply the other integer to current ciphertext
        '''
        # we just need to multiply the give plaintext to both a and b
        mul_a = np.multiply(other, self.a, dtype=np.int32)
        mul_b = np.multiply(other, self.b, dtype=np.int32)
        return LWECiphertext(mul_a, mul_b)
    
    def add_plaintext(self, other:LWEPlaintext):
        '''
            Add the other plaintext to the current ciphertext. This function assumes that the input plaintext message is already encoded to R_q.
        '''
        # we just need to scale other and then add it to b
        addb = np.add(self.b, other.m, dtype=np.int32)
        return LWECiphertext(self.a, addb)

In [6]:
class LWEEncryptionKey:
    '''
        This class implements the LWEEncryption key. The encryption key implements creating the key, which is just a random binary vector or n elements and encrypting a ciphertext form a plaintext.
    '''

    def __init__(self, config:LWEConfig):
        self.config = config
        self.key = np.ndarray # this just initializes key to be an array, which will be then populated
    
    def generate_key(self):
        '''
            This function generates the secret key.
        '''
        self.key = np.random.randint(low=0, high=2, size=self.config.n)

    def encrypt(self, plaintext:LWEPlaintext):
        '''
            Encrypt the plaintext message
        '''
        # generate a and noise
        a = generate_uniform_sample(self.config)
        e = generate_normal_sample(self.config)

        # get b
        b = np.dot(a, self.key) + plaintext.m + e 

        return LWECiphertext(a, b)
    
    def decrypt(self, cipher:LWECiphertext):
        '''
            This function decrypts the ciphertext generated using the same secret key.
        '''
        b = cipher.b
        a = cipher.a

        inter = b - np.dot(a, self.key) # gives m + e, and not just m
        # return LWEPlaintext(inter)

        # round and get the final decrypted output
        final = decode_plaintext(inter, self.config)

        return LWEPlaintext(int(final))

#### Testing encryption

In [ ]:
lweconfig = LWEConfig(p=1<<4, q=1<<32, noise_level=2**-20, dimension=1024)
plain_m = encode_plaintext(-1, lweconfig)
plain_m2 = encode_plaintext(2, lweconfig)

In [10]:
secret_key = LWEEncryptionKey(lweconfig)
secret_key.generate_key()
m_cipher = secret_key.encrypt(plain_m)
m2_cipher = secret_key.encrypt(plain_m2)

In [12]:
m_cipher.a, m_cipher.b

(array([-2078483624,  1592571881,    56415776, ..., -1803974092,
         1464474081,  1138772844], shape=(1024,), dtype=int32),
 np.int64(-35237557257))

In [13]:
m2_cipher.a, m2_cipher.b

(array([  318033945, -1622640825,  1711072075, ...,  -176336818,
         -111803078,  -255263512], shape=(1024,), dtype=int32),
 np.int64(35145243139))

In [14]:
summed = m_cipher.add_ciphertext(m2_cipher)

In [15]:
summed_decrypt = secret_key.decrypt(summed)
summed_decrypt

1

In [11]:
subbed = m_cipher.sub_ciphertext(m2_cipher)
subbed_decrypt = secret_key.decrypt(subbed)
subbed_decrypt

-3

In [12]:
muled = m_cipher.mul_plaintext(np.int32(2))
muled_decrypt = secret_key.decrypt(muled)
muled_decrypt

-2

In [13]:
pplain = encode_plaintext(2, lweconfig)
addp = m_cipher.add_plaintext(pplain)
addp_decrypt = secret_key.decrypt(addp)
addp_decrypt

1

# RLWE
Here, the messages are polyonomials and not integers.

In [7]:
from __future__ import annotations
class Polynomial:
    '''
        This class represents a polynomial in the ring Z_q[x]/(x^n + 1)
        Note that the coefficients here are in increasing order of degrees of x, [constant, x, x**2, ..., x**N]
    '''

    def __init__(self, N, coeffs: np.ndarray):
        self.N = N
        self.coeffs = coeffs

    def __str__(self):
        '''
            Print the ciphertext as an equation.
        '''
        prtstr = ""
        nonzeroct = 0
        for i, c in enumerate(self.coeffs):
            if c:
                if nonzeroct:
                    prtstr += " + "
                prtstr += f"{c}x^{i}"
                nonzeroct += 1

        return prtstr

    def polynomial_const_multiply(self, c:np.int32):
        '''
            Multiply the current polynomial with the given constant integer and return it as a new polynomial.
        '''
        new_p = Polynomial(self.N, np.multiply(c, self.coeffs, dtype=np.int32))
        return new_p
    
    def polynomial_multiply(self, p2: Polynomial):
        '''
            Multiply the current polynomial with the given poylnomial and return the new polynomial.
        '''
        # get the N from p2
        p2N = p2.N
        # multiply and pad the results to have 2N-1 (thats the longest polynomial you can get)
        # the np.polymul function requires the coefficients in decreasing order of powers or x, se we reverse the current coeff array and then reverse the final output's back again.
        poly_prod = np.polymul(self.coeffs[::-1], p2.coeffs[::-1])[::-1]
        # creating the final answer with padded coeffs
        poly_padded = np.zeros(2*self.N - 1, dtype=np.int32)
        poly_padded[:poly_prod.shape[0]] = poly_prod

        # now, taking modulus of the result (poly_padded) wrt x^N + 1
        result = poly_padded[:self.N]
        result[:-1] -= poly_padded[self.N:]
        return Polynomial(self.N, result)
    
    def polynomial_add(self, p2: Polynomial):
        '''
            This function adds a polyonmial to the given polynomial and returns the output as a new polynomial.
        '''
        add_coeffs = np.add(self.coeffs, p2.coeffs, dtype=np.int32)
        return Polynomial(self.N, add_coeffs)
    
    def polynomial_subtract(self, p2: Polynomial):
        '''
            This function subtracts the given polynomial from the current one and returns the new polynomial.
            Note: Its self - p2 and not p2 - self.
        '''
        sub_coeffs = np.subtract(self.coeffs, p2.coeffs, dtype=np.int32)
        return Polynomial(self.N, sub_coeffs)
    
def make_zero_polynomial(N:int):
    '''
        Declare a zero polynomial and return it. Basically a polynomial where all the coefficients are zeros.
    '''
    return Polynomial(N = N, coeffs = np.zeros(N, dtype=np.int32))

def build_monomial(N, i, c):
    '''
        This function builds a monomial c * x^i in the ring R[x]/(x^N + 1)
        NOTE: Rememeber that the coeffs are in increasing order of powers of x. [constant, x, x**2, ..., x**N]
    '''
    coeffs = np.zeros(N, dtype=np.int32)

    # find k such that 0 <= i + k*N < N
    i_mod_n = i % N
    k = (i_mod_n - i)//N

    # if k is odd then the monomial gets a negative sign because
    # x^i = (-1)^k + x^(i + k*N) = (-1)^k + x^(i % N)
    sign = 1 if k % 2 == 0 else -1
    coeffs[i_mod_n] = sign * c

    return Polynomial(N, coeffs)

In [8]:
class RLWEConfig:
    '''
        This class defines the RLWE config.
    '''
    def __init__(self, N, p, q, sigma):
        self.N = N
        self.p = p
        self.q = q
        self.sigma = sigma

In [9]:
class RLWEPlaintext:
    '''
        This class implements the RLWE Plaintext. This just stores the encoded message polynomial. Functionally, it isnt really needed but Ive just added it to make my code consistent with how TFHE libraries work.
    '''
    def __init__(self, p: Polynomial, config: RLWEConfig):
        self.p = p
        self.config = config

In [10]:
# utility functions for RLWE
def encode_rlwe(p: Polynomial, config: RLWEConfig):
    '''
        This function encodes the cleartext polynomial and returns it as a Plaintext.
    '''
    # get the delta
    delta = config.q//config.p
    # scale each coefficient of the polynomial by multiplying it with delta
    scaled_coeffs = np.multiply(p.coeffs, delta, dtype=np.int32)
    return RLWEPlaintext(Polynomial(p.N, scaled_coeffs), config=config)

def decode_coeff(c, delta, p):
    '''
        This function decodes a specific coefficient within the polynomial.
    '''
    # divide by delta and round
    decoded_c = int(np.rint(c/delta))
    # calculate decoded_c mod p and you get the final decoded value
    out = ((decoded_c + (p//2)) % p) - (p//2)
    return out

def decode_rlwe(p: Polynomial, config: RLWEConfig):
    '''
        This function decodes the given polynomial to remove the noise and get the final decrypted cleartext.
    '''
    # calcualte the delta
    delta = config.q//config.p
    raw_coeffs = p.coeffs
    # decoded each coeff and get the output polynomial
    decoded_coeffs = np.array([decode_coeff(c, delta,config.p) for c in raw_coeffs])

    return Polynomial(config.N, decoded_coeffs)

def rlwe_generate_uniform_sample(config: RLWEConfig) -> Polynomial:
    '''
        This function generates a polynomial of the degree specified in the config where each coefficient is an integer sampled from a uniform random distribution.
    '''
    # get the limits
    minval = -config.q//2
    maxval = config.q//2

    # generate N values between the min (inclusive) and max (exclusive)
    coeffs = np.random.randint(minval, maxval, size=config.N, dtype=np.int32)

    return Polynomial(config.N, coeffs=coeffs)

def rlwe_generate_normal_sample(config: RLWEConfig) -> Polynomial:
    '''
        This function generates a polynomial of the degree as per the config where each coefficient is drawn from a normal distribution with mean 0 and standard distribution given in the config.
    '''
    coeffs = np.int32(np.multiply(np.random.normal(0, config.sigma, size=config.N), config.q//2))
    return Polynomial(config.N, coeffs)

In [11]:
class RLWECiphertext:
    '''
        This class implements the RLWE ciphertext.
    '''
    def __init__(self, a: Polynomial, b: Polynomial, config: RLWEConfig):
        self.config = config
        self.a = a
        self.b = b

    def add_ciphertext(self, other: RLWECiphertext):
        '''
            Add another ciphertext <other> to the current ciphertext and return the result as a new ciphertext. Two add two ciphertexts, you just add their respective a and b polynomials.
        '''
        # adding a and b, and creating a new ciphertext
        added_a = self.a.polynomial_add(other.a)
        added_b = self.b.polynomial_add(other.b)

        return RLWECiphertext(added_a, added_b, self.config)
    
    def sub_ciphertext(self, other: RLWECiphertext):
        '''
            Subtract another ciphertext <other> from the current one and return the result as a new ciphertext. This operation subtracts <other> from the <self> and not the other way round.
        '''
        # subtracting a and b and returning a new ciphertext
        subbed_a = self.a.polynomial_subtract(other.a)
        subbed_b = self.b.polynomial_subtract(other.b)

        return RLWECiphertext(subbed_a, subbed_b, self.config)
    
    def mul_plaintext(self, cx: RLWEPlaintext):
        '''
            This function multiplys the given plaintext to self and returns the product as a new ciphertext.
        '''
        # for this, you multiply both a and b with the plaintext
        mul_a = self.a.polynomial_multiply(cx.p)
        mul_b = self.b.polynomial_multiply(cx.p)

        return RLWECiphertext(mul_a, mul_b, self.config)
        

In [12]:
class RLWEEncryptionKey:
    '''
        This class defines the RLWE encryption/secret key and performs encryption and decryption of ciphertexts.
    '''
    def __init__(self, config:RLWEConfig):
        self.config = config
        self.key = Polynomial # this time, the key is also a polynomial.

    def generate_key(self):
        '''
            Generate the key. The key is a random polynomial of degree N with binary coefficients.
        '''
        self.key = Polynomial(
            N=self.config.N,
            coeffs = np.random.randint(
                low = 0,
                high = 2,
                size = self.config.N,
                dtype = np.int32
            )
        )
    
    def encrypt(self, mx: RLWEPlaintext):
        '''
            Encrypt the input Polynomial p into an RLWE Ciphertext.
        '''
        # generate a and noise e
        a = rlwe_generate_uniform_sample(self.config)
        e = rlwe_generate_normal_sample(self.config)

        # encrypting
        a_times_s = a.polynomial_multiply(self.key) # <a, s>
        as_plus_m = a_times_s.polynomial_add(mx.p) # <a, s> + m (m is already encoded, which is why I wrote m and not delta * m)
        b = as_plus_m.polynomial_add(e)

        return RLWECiphertext(a, b, self.config)
    
    def decrypt(self, cx: RLWECiphertext) -> Polynomial:
        '''
            Decrypt the given ciphertext, decode it and return the cleartext polynomial.
        '''
        # separate a and b
        ca = cx.a
        cb = cx.b
        # subtracting <a, s> from b
        subbed = cb.polynomial_subtract(ca.polynomial_multiply(self.key))
        # decode: divide by delta to remove the noise and take mod wrt p
        decoded = decode_rlwe(subbed, self.config)

        return decoded

In [13]:
def get_trivial_rlwe_ciphertext(mx: Polynomial, config: RLWEConfig):
    '''
        This function returns the trivial RLWE ciphertext for the given config. Trivial RLWE ciphertext is basically just an RLWE ciphertext where a is just a zero polynomial and b is polynomial itself.
        NOTE: Im not very sure if we should encode the input polynomial, meaning, if we take a cleartext input or plaintext input.
    '''
    # sanity check
    if len(mx.coeffs) != config.N:
        raise ValueError("The polynomial has a degree thats different from the degree provided in the config file. Please ensure that theyre the same.")
    # creating a
    a = make_zero_polynomial(config.N)
    return RLWECiphertext(a = a, b = mx, config=config)

In [14]:
def lwe_to_rlwe_key(lwe_key: LWEEncryptionKey, config: RLWEConfig):
    '''
        This function creates an RLWE key for/from the given RLWE key.
    '''
    # creating a polynomial with the coeffs as the lwe_key
    if lwe_key.config.n != config.N:
        raise ValueError("The size of LWEKey and degree of RLWEConfig do not match.")
    key_poly = Polynomial(N=config.N, coeffs=lwe_key.key)
    rlwe_key = RLWEEncryptionKey(config=config)
    rlwe_key.key = key_poly

    return rlwe_key

### Trying encryption

In [15]:
# config
config = RLWEConfig(N=1024, sigma=2**(-24), p = 16, q=2**32)

In [27]:
# generating the encryption key
secret_key = RLWEEncryptionKey(config=config)
secret_key.generate_key()

In [38]:
# encrypting a polynomial, say 4x^2
clear_poly = build_monomial(config.N, i=2, c=2)
clear_poly2 = build_monomial(config.N, i=3, c=3)
# clear_poly = make_zero_polynomial(config.N)

In [39]:
# encoding the polynomial
plain_poly = encode_rlwe(clear_poly, config)
plain_poly2 = RLWEPlaintext(config=config, p=clear_poly2)

In [40]:
# encrypt the ciphertext
cipher_poly = secret_key.encrypt(plain_poly)

In [41]:
# decrypt the ciphertext
decrypt_poly = secret_key.decrypt(cipher_poly)
print(decrypt_poly)

2x^2


In [42]:
mul_poly = cipher_poly.mul_plaintext(plain_poly2)

In [43]:
print(secret_key.decrypt(mul_poly))

6x^5


In [44]:
print(clear_poly.polynomial_multiply(clear_poly2))

6x^5


# GSW Encryption

This will be used in implementing the multiplexer function which will be used in bootstrapping, and implementing ciphertext multiplication between a GSW ciphertext and an RLWE ciphertext is possible.

In [16]:
# functions to get the p-bit representation of x (I think x is going to be an RLWE ciphertext in practice, this will be confirmed later)
from typing import Sequence

def get_k(q, p):
    '''
        Given q (ciphertext modulus) and p (not plaintext modulus, but a different parameter specifically for base p representation used for ciphertext-ciphertext multiplication of an RLWE and GWE cipherterxts), get the value of k, which is the sequence length of base-p representation of any input.
    '''
    return np.int32(np.log2(q)/np.log2(p))

def convert_array_to_base_p(a: np.ndarray, p, q):
    '''
        This function converts each element into a its base p-representation, given the values of q (ciphertext modulus) and p (not plaintext modulus). Note that the representation for each element in a will be a sequence of a fixed length.
    '''
    log_p = np.log2(p).astype(np.int32)
    k = get_k(q = q, p = p)
    p_half = p//2
    offset = p_half * np.sum([p**i for i in range(k)])
    mask = p - 1 # this in binary is exactly log_p ones.

    a_offset = np.uint32(a + offset) # need to set this to uint 32 as the the p-bit representation will be calculated from unsigned integers and then converted to signed
    # return a_offset
 
    output = [] # this will store the output base-p representation for each element
    # NOTE: output is a list of k numpy arrays. Each numpy array stores the ith p-bit representation of the corresponding original input.
    for i in range(k):
        output.append(
            (np.right_shift(a_offset, i * log_p) & mask).astype(np.int32) - np.int32(p_half)
            )
        
    return output

def convert_base_p_to_int_array(a: Sequence[np.array], p: np.int32):
    '''
        This function takes as input the p-bit representations of an array, and converts it back into integer. a should be a list or a iterator of ndarrays where each ndarray contains the ith p-bit representation of the integer.
    '''
    # calculate x from the representation: x = x_0 + x_1 p + ... + x_k-1 p^k-1
    decimal_repr = np.zeros(len(a[0]), dtype=np.int32)
    for i in range(len(a)):
        decimal_repr += (a[i] * (p**i)).astype(np.int32)
    return decimal_repr

In [17]:
# testing the function
arr = np.array([1000, 256, 3])
p_bitrepr = convert_array_to_base_p(arr, 2**8, 2**32)
p_bitrepr

[array([-24,   0,   3], dtype=int32),
 array([4, 1, 0], dtype=int32),
 array([0, 0, 0], dtype=int32),
 array([0, 0, 0], dtype=int32)]

In [31]:
# converting back to decimal repre
dec_arr = convert_base_p_to_int_array(p_bitrepr, 2**8)
dec_arr

array([1000,  256,    3], dtype=int32)

In [18]:
# defining base p representation functions for polynomials
def get_base_p_from_polynomial(poly: Polynomial, p: np.int32, q: np.int32) -> Sequence[Polynomial]:
    '''
        This function converts each coefficient of the coefficients of the given polynomial and returns them as a set of polynomials (sequence of the polynomials is very important).
    '''
    coeffs = poly.coeffs
    # converting coeffs into base p representation
    coeffs_base_p = convert_array_to_base_p(coeffs, p, q)
    # create a polynomial for each array
    base_p_poly = [Polynomial(N=poly.N, coeffs = ic) for ic in coeffs_base_p]

    return base_p_poly

def get_poly_from_base_p(base_p_polys: Sequence[Polynomial], p) -> Polynomial:
    '''
        This function calculates the polynomial from the given base p representation polynomials.
    '''
    # get all the coefficients from the sequence
    base_p_coeffs = [p.coeffs for p in base_p_polys]
    # get the integer coeffs from the list
    int_coeffs = convert_base_p_to_int_array(base_p_coeffs, p)
    return Polynomial(N=base_p_polys[0].N, coeffs = int_coeffs)

In [19]:
# testing the function
tst_poly = Polynomial(4, [1, 1000, 256, -3])
# getting the base p representation
base_p_polys = get_base_p_from_polynomial(tst_poly, 2**8, 2**32)
# getting the intgeer representation back
int_poly = get_poly_from_base_p(base_p_polys=base_p_polys, p=2**8)
print(int_poly)

1x^0 + 1000x^1 + 256x^2 + -3x^3


In [20]:
class GSWConfig:
    '''
        This class implements the GSW Config. It takes the RLWE config, as most of the configurations are identical. Additionally, it also takes gsw_p, which will be used for the base p representation.
    '''
    def __init__(self, rlwecfg: RLWEConfig, gsw_p: int):
        self.rwlecfg = rlwecfg
        self.gsw_p = gsw_p

In [21]:
class GSWPlaintext:
    '''
        This class represents the gsw plaintext.
    '''
    def __init__(self, config: GSWConfig, message: Polynomial):
        self.config = config
        self.p = message

In [22]:
class GSWCiphertext:
    '''
        This class implements the gsw ciphertext.
    '''
    def __init__(self, config: GSWConfig, polys: Sequence[RLWECiphertext]):
        self.config = config
        self.polys = polys

In [23]:
class GSWEncryptionKey:
    '''
        This class implements the GSW encryption key and methods to encrypt and (possibly) decrypt a gsw ciphertext.
    '''
    def __init__(self, config: GSWConfig):
        self.config = config
        self.key = Polynomial # this is just going to be the RLWE encryption key s(x)
        self.rlwe_key = RLWEEncryptionKey

    def generate_key_from_rlwe_key(self, rlwe_key: RLWEEncryptionKey):
        '''
            This function creates the GSWEncryptionKey from the RLWEEncryption key. To be honest, its just the RLWEEncryption key as it is, but for the sake of structure of code and to keep it consistent with other classes. Its a little silly to save both the rlwe_key object and the key polynomail as well, but eh...
        '''
        self.key = rlwe_key.key
        self.rlwe_key = rlwe_key

    def get_rlwe_key_from_gsw(self):
        '''
            This function returns the rlwe key from the current gsw_key. Again, its the same, but consistency, formality, something, something...
        '''
        rlwe_key = RLWEEncryptionKey(config=self.config.rwlecfg)
        rlwe_key.key = self.key
        return rlwe_key
    
    
    def encrypt(self, fx: Polynomial):
        '''
            Encrypt a given polynomial as a GSWCiphertext.
        '''
        # first, we need to get the value of k
        k = get_k(self.config.rwlecfg.q, self.config.gsw_p)
        gsw_p = np.int32(self.config.gsw_p) # i dont want to write the whole thing
        # make Z: the zero encryption array of shape (2k, 2), note that each row is a RLWE ciphertext, the 2 just means (a(x), b(x))
        zero_poly = make_zero_polynomial(self.config.rwlecfg.N)
        zero_poly_plain = encode_rlwe(zero_poly, self.config.rwlecfg) # while for 0 config, the coefficients for both the plaintext and ciphertext are going to be the same, the encryption for RLWE only accepts an RLWEPlaintext and not a Polynomial
        Z = [
            self.rlwe_key.encrypt(zero_poly_plain) for _ in range(2*k)
        ] # this has shpae (2k, 2)
        # create the GSW ciphertext
        # its created using the equation: fx * B_p + Z
        # i will implicitly divide the Z matrix in two halves 
        # here, ill iterate for k times, and for each iteration, the corresponding index's RLWE of the first half's a gets fx * (p**i) added and the corresponding index of the second half's b gets the same added
        for i in range(k):
            # multiply fx with the correct power of p
            scaled_fx = fx.polynomial_const_multiply(gsw_p**i)
            # add it to first half's a
            Z[i].a = Z[i].a.polynomial_add(scaled_fx)
            # add it to second half's b
            Z[i + k].b = Z[i + k].b.polynomial_add(scaled_fx)

        return GSWCiphertext(self.config, Z)

In [24]:
def ciphertext_multiplication(G: GSWCiphertext, R: RLWECiphertext) -> RLWECiphertext:
    '''
        This function performs the ciphertext-ciphertext multiplication between G and R. G is of shape (2k, 2), basically 2k RLWE Ciphertexts and each Rlwe ciphertext is (a, b).
        So a little explanation because this function looks a little confusing (but in reality it really isnt once you look at it hard enough keeping in mind how the multiplication is actually done). Im going to define in the docstring how does this operation actually works. 
        NOTE TO SELF: If its been long since this code was studied last, I practically beg you to please read the thing after so in its entirety. Otherwise, there is a real chance that something in your brain explodes (or it starts to hurt) and I dont want that because I care for you :)
        Also, I know that im calling using the term vector of polynomials, which might sound wierd becasue a polynomial itself is a vector of coefficients, so a vector of polynomials is just a higher dimension matrix, but in FHE, it seems that polynomials are referred to as a single entity, adressed like constant numbers, so Im just keeping it consistent.
        So:
        First, G is a series of 2k RLWECiphertexts. Essentially, theyre all zero ciphertexts to which fx (the polynomial with which we actually want to achieve multiplication with R) is added. For the first half of G [0:k-1], fx is added to a(x) of each RLWE 'row', and is added to b(x) for the second half [k: 2k-1].
        Second, a(x) and b(x) of R are converted to corresponding base-p representation. Now R was basically a vector/sequence/whatever of 2 polynomials, but now it becomes a vector of 2k polynomials (k polynomials for base p representation of a(x) and b(x)). 
        To perform c-c multiplication, you basically do something akin to matrix multiplication, (Basep(a(x)), Basep(b(x))) @ G, which gives you a single rlwe polynomial, and its shape will be (1, 2), a single (a(x), b(x)) ciphertext. Here, each multiplication is polynomial multiplication followed by polynomial addition for matmul.
    '''
    gsw_config = G.config
    rlwe_config = R.config
    # first get the base p representation for R, for both a(x) and b(x) and create it into a sequence of 2k polynomials
    R_base_p_repr = get_base_p_from_polynomial(R.a, p=gsw_config.gsw_p, q=rlwe_config.q) + get_base_p_from_polynomial(R.b, p=gsw_config.gsw_p, q=rlwe_config.q) # the output of the function is a list so + just appends
    
    # so matmul here is implemented using a for loop, first, an RLWECiphertext that is basically all zeros is initialized and then the corresponding products a.G and b.G for each R_base_p_repr is added to a and be of this new RLWECiphertext respectively
    res_rlwe_cipher = RLWECiphertext(
        config = rlwe_config,
        a = make_zero_polynomial(rlwe_config.N),
        b = make_zero_polynomial(rlwe_config.N)
    )

    for i, rp in enumerate(R_base_p_repr):
        # multiply the current r to both a and b of the current row of g.a
        a_mul = rp.polynomial_multiply(G.polys[i].a)
        # adding
        res_rlwe_cipher.a = res_rlwe_cipher.a.polynomial_add(a_mul)

        # doing the same for b
        b_mul = rp.polynomial_multiply(G.polys[i].b)
        # adding
        res_rlwe_cipher.b = res_rlwe_cipher.b.polynomial_add(b_mul)

    return res_rlwe_cipher

In [30]:
gsw_config = GSWConfig(config, 2**8)
rlwe_key = RLWEEncryptionKey(config)
rlwe_key.generate_key()
gsw_key = GSWEncryptionKey(gsw_config)
gsw_key.generate_key_from_rlwe_key(rlwe_key)

In [40]:
config.p

16

In [41]:
# creating gsw polynomial
fx = build_monomial(gsw_config.rwlecfg.N, 3, 2)
fx_gsw_cipher = gsw_key.encrypt(fx)

# creating rlwe polynomial
f2x = build_monomial(config.N, i=2, c=1)
plain_poly = encode_rlwe(f2x, config)
cipher_poly = rlwe_key.encrypt(plain_poly)

In [42]:
mult_cipher = ciphertext_multiplication(fx_gsw_cipher, cipher_poly)

In [43]:
decrypt_poly = rlwe_key.decrypt(mult_cipher)
print(decrypt_poly) # correct multiplication

2x^5


### Implementing Homomorphic multiplexer using these operations.

In [ ]:
def cmux(b: GSWCiphertext, l0: RLWECiphertext, l1:RLWECiphertext) -> RLWECiphertext:
    '''
        This function implements the multiplexer for ciphertexts.
        The homomorphic multiplexer operation homomorphically computes: b(l1 - l0) + l0.
        So its basically chaining the following operations: CAdd(CMul(b, CSub(l1, l0)), l0)
        the function returns l0 is b == 0, else l1.
    '''
    # calculating l1 - l0
    subbed = l1.sub_ciphertext(l0)
    # calculating product of subbed and b
    prod = ciphertext_multiplication(b, subbed)
    # adding l0 to get the cmux result
    mux_cipher = prod.add_ciphertext(l0)

    return mux_cipher

In [45]:
# The selector bit is b=1
b = build_monomial(c=1, i=0, N=config.N)

# The lines are: l_0(x) = x, l_1(x) = 2x
l0 = build_monomial(c=1, i=1, N=config.N)
l1 = build_monomial(c=2, i=3, N=config.N)
l0_plaintext = encode_rlwe(l0, config)
l1_plaintext = encode_rlwe(l1, config)

# encrypt them
b_gsw_cipher = gsw_key.encrypt(b)
l0_cipher = rlwe_key.encrypt(l0_plaintext)
l1_cipher = rlwe_key.encrypt(l1_plaintext)

In [46]:
cmux_res = cmux(b_gsw_cipher, l0_cipher, l1_cipher)

In [47]:
cmux_decrypt = rlwe_key.decrypt(cmux_res)
print(cmux_decrypt)

2x^3


# BlindRotate and Bootstrapping Keys

In [26]:
class BootstrapKey:
    '''
        This class implements the bootstrap keys and the methods associated with it, like BlindRotate.
    '''
    def __init__(self, config: GSWConfig):
        self.config = config
        self.key = Sequence[GSWCiphertext] # each component (s_i) from the LWE key will be a GSW ciphertext

    def generate_bootstrap_key(self, lwe_key: LWEEncryptionKey, gsw_key: GSWEncryptionKey):
        '''
            generate the bootstrapping keys. The bootstrapping keys are just GSW encryption of each component of the LWE_key using the gsw_key provided.
            NOTE TO SELF: Im debating whether to make the lwe_key and gsw_key members of this class or not. Right now I wont, but if there are other operations that require these keys in addition to the bootstrap keys, I will.
        '''
        bs_keys = [] # this will store the keys
        # looping across each lwe key component and encrypting it
        for si in lwe_key.key:
            # first, create a monomial with the si being constant
            si_monomial = build_monomial(self.config.rwlecfg.N, 0, si)
            si_gsw = gsw_key.encrypt(si_monomial)
            bs_keys.append(si_gsw)
        self.key = bs_keys

    def blindrotate(self, FX: RLWECiphertext, I: LWECiphertext) -> RLWECiphertext:
        '''
            This function implements the Rotate(I, FX) function homomorphically. This function, given i, and a polynomial f(x) (I = LWE(i), FX = RLWE(f(x)), calculates x^i . f(x) as an RLWECiphertext.
        '''
        # scale the lwe ciphertext 
        scaled_I_a = np.int32(np.rint(np.multiply(I.a, ((2 * self.config.rwlecfg.N)/ self.config.rwlecfg.q)))) 
        scaled_I_b = np.int32(np.rint(np.multiply(I.b, ((2 * self.config.rwlecfg.N)/ self.config.rwlecfg.q)))) # NOTE: This is going to be a constant since this is an LWE ciphertext

        # rotate operation
        # initialize g0
        xb_poly = build_monomial(self.config.rwlecfg.N, scaled_I_b, 1)
        xb_plain = RLWEPlaintext(xb_poly, FX.config)        
        rotated_poly = FX.mul_plaintext(xb_plain) # this will updated every timestep
        # loop N times to finally get the rotated output
        for i, ai in enumerate(scaled_I_a):
            # get g_i-1 (x) * x^a_i
            poly = RLWEPlaintext(build_monomial(self.config.rwlecfg.N, -ai, 1), self.config.rwlecfg) # just converting it to a plaintext because the plaintext multiply in RLWE wants a plaintext
            right = rotated_poly.mul_plaintext(poly)
            # update rotated_poly using the cmux operation
            rotated_poly = cmux(self.key[i], rotated_poly, right)
        
        return rotated_poly

Testing this implementation

In [31]:
# --- SETTING UP THE KEYS ---
# generate lwe key
lwe_key = LWEEncryptionKey(config=lweconfig)
lwe_key.generate_key()
# get rlwe key from this
rlwe_key = lwe_to_rlwe_key(lwe_key, config)
# get the gsw key
gsw_key = GSWEncryptionKey(gsw_config)
gsw_key.generate_key_from_rlwe_key(rlwe_key)
# get the bootstrap key
boostrap_key = BootstrapKey(gsw_config)
boostrap_key.generate_bootstrap_key(lwe_key=lwe_key, gsw_key=gsw_key)

In [32]:
# generating a polynomial to rotate:
# we are going to rotate the polynomial: -1 -x -x**2 - ....- x**(N/2-1) + x^(N/2)+ .... + x^(N-1)
# the reason why we are rotating such a large polynomial is just because this makes it easy to know that its rotated since N is so big (1024 here)
# original coeffs: [-1, -1, ..., -1 (until the 1023 index), 1, 1, ...., 1]
N = config.N
fx = Polynomial(N=N, coeffs=np.ones(N, dtype=np.int32))
fx.coeffs[: N // 2] = -1

# encrypt fx
fx_plain = encode_rlwe(fx, config)
fx_rlwe = rlwe_key.encrypt(fx_plain)

In [ ]:
# for this test, we rotate the ciphertext by (3/4) N.
# remember that p = 16 (2^4) and q = 2^31, so for the scaled i to be (3/4)N, we need to have i be 6. WHY:
# encoding 6 makes it (6q/16), and multiplying it with (2q/N) gives (3N/4)
i_plain = encode_plaintext(6, lweconfig)
i_lwe = lwe_key.encrypt(i_plain)

In [34]:
# rotating
rotated_rlwe = boostrap_key.blindrotate(fx_rlwe, i_lwe)

In [35]:
rotated_clear = rlwe_key.decrypt(rotated_rlwe)
print(rotated_clear)

1x^0 + 1x^1 + 1x^2 + 1x^3 + 1x^4 + 1x^5 + 1x^6 + 1x^7 + 1x^8 + 1x^9 + 1x^10 + 1x^11 + 1x^12 + 1x^13 + 1x^14 + 1x^15 + 1x^16 + 1x^17 + 1x^18 + 1x^19 + 1x^20 + 1x^21 + 1x^22 + 1x^23 + 1x^24 + 1x^25 + 1x^26 + 1x^27 + 1x^28 + 1x^29 + 1x^30 + 1x^31 + 1x^32 + 1x^33 + 1x^34 + 1x^35 + 1x^36 + 1x^37 + 1x^38 + 1x^39 + 1x^40 + 1x^41 + 1x^42 + 1x^43 + 1x^44 + 1x^45 + 1x^46 + 1x^47 + 1x^48 + 1x^49 + 1x^50 + 1x^51 + 1x^52 + 1x^53 + 1x^54 + 1x^55 + 1x^56 + 1x^57 + 1x^58 + 1x^59 + 1x^60 + 1x^61 + 1x^62 + 1x^63 + 1x^64 + 1x^65 + 1x^66 + 1x^67 + 1x^68 + 1x^69 + 1x^70 + 1x^71 + 1x^72 + 1x^73 + 1x^74 + 1x^75 + 1x^76 + 1x^77 + 1x^78 + 1x^79 + 1x^80 + 1x^81 + 1x^82 + 1x^83 + 1x^84 + 1x^85 + 1x^86 + 1x^87 + 1x^88 + 1x^89 + 1x^90 + 1x^91 + 1x^92 + 1x^93 + 1x^94 + 1x^95 + 1x^96 + 1x^97 + 1x^98 + 1x^99 + 1x^100 + 1x^101 + 1x^102 + 1x^103 + 1x^104 + 1x^105 + 1x^106 + 1x^107 + 1x^108 + 1x^109 + 1x^110 + 1x^111 + 1x^112 + 1x^113 + 1x^114 + 1x^115 + 1x^116 + 1x^117 + 1x^118 + 1x^119 + 1x^120 + 1x^121 + 1x^122 + 1x^

In [36]:
assert rotated_clear.coeffs[0] == 1
assert rotated_clear.coeffs[N // 2] == -1
assert rotated_clear.coeffs[-1] == -1

# Sample Extraction

In [41]:
def extract_sample(i: int, FX: RLWECiphertext) -> LWECiphertext:
    '''
        This function extracts the ith coefficient of the polynomial encrypted in the input FX and returns the ith coefficient as an LWE Polynomial. It calculates the ai (a for the LWE ciphertext) using Conv(a(x), i) and the ith coefficient of b(x). The resulting LWE ciphertext is encrypted using the LWE key obtained from the RLWE key (you can convert them into one another).
    '''
    # first get the conv(a(x), i)
    # conv(a(x), i) is just going to be coeffs of a(x), but 'rotated' around x.
    # that means, if a(x) was [a0, a1, a2, ...., a(N-1)]
    # then conv(a(x), i) is [ai, a_i-1, ..., a1, a0, -aN-1, -aN-1, ..., -ai+1]
    ax = FX.a.coeffs
    conv_ax = np.hstack([
        ax[:i+1][::-1],
        -1 * ax[i+1:][::-1]])
    bi = FX.b.coeffs[i] # just need the ith coefficient's b

    # creating an LWECiphertext for this
    return LWECiphertext(
        a = conv_ax,
        b = bi
    )

In [42]:
# testing
fx = build_monomial(config.N, 1, -2) # 2x
# encrypting
fx_plain = encode_rlwe(fx, config)
fx_rlwe = rlwe_key.encrypt(fx_plain)

In [ ]:
# extracting the index at i=1
extracted_lwe = extract_sample(i = 1, FX = fx_rlwe)
# decrypting
# NOTE: Im using the lwe_key object here because the rlwe key used for encryption above was created from this lwe_key (look at the blind rotate section of the notebook for more context)
plain_lwe = lwe_key.decrypt(extracted_lwe)
plain_lwe

-2

In [40]:
config.p, lweconfig.p

(16, 16)

# Bootstrapping

Bootstrapping in TFHE implements a specific function, and is configured for it. For this notebook, the end-goal is to compute a NAND operation between two inputs (LWE Ciphertexts). The NAND operation can be implemented using the following way.
- First, we define a function $F(l_0, l_1) = Encode(-3) - l_0 - l_1$. This function returns an lwe ciphertext.
- $F(l_0, l_1)$'s output will be used as an input to the Step function, which is defined as: $Step(x) = 0$ if $ -q/4 < x \leq q/4$; $Encode(2)$ otherwise. Also, this step function will be implemented by bootstrapping.
- The output of the step function gives is the result of $\text{NAND}(l_0, l_1)$.

__How does Bootstrapping work:__ <br>
First, you need to declare something called as a test polynomial $t(x)$, which is just a polynomial of degree N-1, that looks like:
$$t(x) = -1 -x -x^2 - ... - x^{\frac{N}{2} - 1} - x^{\frac{N}{2}} + x^{\frac{N}{2} + 1} + ... + x^{N-1}$$
t(x) is scaled and then encrypted to a trivial RLWE Ciphertext: $R = \text{Enc}^{RLWE}_{s(s)} (\frac{\text{Encode}(2)}{2} \cdot t(x))$ <br><br>
Let:
- i = LWE encryption of the input
- B = Trivial LWE encryption of $\frac{Encode(2)}{2}$<br>

Then, bootstrap is the following operation: Bootstrap(i) = CAdd(B, ExtractSample(0, BlindRotate(i, R))).<br>
Please note that this specific arrangement of bootstrap is (most likely) specific to calculating the step function for the nand operation.

In [ ]:
def bootstrap(i: LWECiphertext, bsk: BootstrapKey, R:RLWECiphertext, scale_lwe: LWECiphertext):
    '''
        This function implements the bootstrap function to calculate the step operation defined above for the input i. The bsk is the bootstrapping key.
        This function returns 0 if i is in (-q/4, q/4], else, it returns an lwe encryption of scale.
        NOTE [IMPORTANT]: Note that if scale is the cleartext np.int32 value, scale_lwe encrypts Encode(scale)/scale.
    '''
    # blind rotate R
    rotated_rlwe = bsk.blindrotate(R, i)
    # extract sample to get the coefficient of constant
    coeff_lwe = extract_sample(0, rotated_rlwe)
    # adding scale_lwe to the output to get the offset-ed result of bootstrapping
    result_lwe = scale_lwe.add_ciphertext(coeff_lwe)

    return result_lwe

In [107]:
def generate_tx(N) -> Polynomial:
    '''
        This function generates the polynomial i(t).
    '''
    # building the polynomial
    tx_coeffs = np.ones(N, dtype=np.int32)
    # updating the first half to be -1
    tx_coeffs[:N//2] = -1
    tx_poly = Polynomial(rlwe_key.config.N, tx_coeffs)
    return tx_poly

In [108]:
def homomorphic_nand(b0: LWECiphertext, b1: LWECiphertext, constCipher: LWECiphertext, scale_lwe: LWECiphertext, txR:RLWECiphertext, bsk:BootstrapKey) -> LWECiphertext:
    '''
        This function implements the nand operation between two inputs b0 and b1 homomorphically.
        The way it does it is by first evaluating the value of F(b0, b1) = LWE(-3) - b0 - b1 and then running the bootstrap that implements the step function which evaluates the final nand function.
        Remember that 0 is equivalent to F while 2 is equivalent to T here. The output will be an LWE ciphertext encrypting 0 or 2 as well.
        constCipher is just the LWE Encryption of -3.
    '''
    # calculating constCipher - b0 - b1
    cc_minus_b0 = constCipher.sub_ciphertext(b0)
    F_lwe = cc_minus_b0.sub_ciphertext(b1)
    # call bootstrapping on this
    nand_lwe = bootstrap(F_lwe, bsk, txR, scale_lwe)

    return nand_lwe

In [109]:
def create_trivial_rlwe_ciphertext(mx: Polynomial, config: RLWEConfig):
    '''
        This function creates a trivial RLWE ciphertext. A trivial RLWE ciphertext is one where the polynomial is one where b is the polynomial itself while a is a zero polynomial.
    '''
    a = make_zero_polynomial(config.N)
    return RLWECiphertext(a, mx, config)

def create_trivial_lwe_ciphertext(m: LWEPlaintext, config: LWEConfig):
    '''
        This function creates a trivial LWE ciphertext, which is an LWECiphertext with a being a vector of zeros and b being the plaintext itself.
    '''
    a = np.zeros(config.n, dtype=np.int32)
    return LWECiphertext(a, m.m)

Testing the functions.

In [110]:
# declaring the configs
lweconfig = LWEConfig(p=1<<3, q=1<<32, noise_level=2**-20, dimension=1024)
rlweconfig = RLWEConfig(N=1024, sigma=2**(-24), p = 1<<3, q=2**32)
gswconfig = GSWConfig(rlweconfig, 2**8)
# NOTE: While its a good idea to have sigma/noise_level the same for lwe and rlwe, but here, im going to use the lwe key to generate rlwe key, so the noise_level of lwe will be used everywhere so its okay
# generating the keys
lwe_key = LWEEncryptionKey(lweconfig)
lwe_key.generate_key()
# get the rlwe key from this
rlwe_key = lwe_to_rlwe_key(lwe_key, rlweconfig)
# get the gsw key
gsw_key = GSWEncryptionKey(gsw_config)
gsw_key.generate_key_from_rlwe_key(rlwe_key)
# get the bootstrap key
bsk =BootstrapKey(gsw_config)
bsk.generate_bootstrap_key(lwe_key=lwe_key, gsw_key=gsw_key)

In [119]:
# bootstrapping
# first, we need to create the test polynomial and encrypt it
tx = generate_tx(rlweconfig.N)
# create Encode(2)
scale_plain = encode_plaintext(2, lweconfig)
# we are actually working with scale_plain//2
scale_plain.m = scale_plain.m//2
# use it to encrypt tx as a trivial rlwe ciphertext
tx_rlwe = create_trivial_rlwe_ciphertext(tx.polynomial_const_multiply(scale_plain.m), rlweconfig)
# create trivial lwe ciphertext for scale_plain
scale_lwe = create_trivial_lwe_ciphertext(scale_plain, lweconfig)

# create encrypted inputs: 0==false, 2==true
l0_plain = encode_plaintext(2, lweconfig) # true or false
l1_plain = encode_plaintext(2, lweconfig) # false or false
# ciphertexts
l0_cipher = lwe_key.encrypt(l0_plain)
l1_cipher = lwe_key.encrypt(l1_plain)
# constant cipher
const_plain = encode_plaintext(-3, lweconfig)
const_cipher = lwe_key.encrypt(const_plain)

nand_cipher = homomorphic_nand(l0_cipher, l1_cipher, const_cipher, scale_lwe, tx_rlwe, bsk)

In [120]:
# decrypting the nand_cipher
nand_clear = lwe_key.decrypt(nand_cipher)
nand_clear

0